# AiXbio — Notebook 1: ESM-2 Embeddings + ProteinMPNN Redesign
Extracts mean-pooled hidden states from ESM-2 650M at 6 layers; runs ProteinMPNN redesigns.

In [1]:
import os, json, warnings, subprocess
from pathlib import Path
import numpy as np
import torch
from Bio import SeqIO, pairwise2
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LAYERS = [1, 9, 18, 24, 30, 33]
print(f'Device: {DEVICE}')

mmseqs_dir = str(Path('mmseqs/bin').resolve())
if mmseqs_dir not in os.environ.get('PATH', ''):
    os.environ['PATH'] = mmseqs_dir + ':' + os.environ.get('PATH', '')

Device: cuda


/home/ubuntu/.local/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Debug: Inspect ProteinMPNN Output Structure

In [2]:
# ── Check what ProteinMPNN actually wrote ─────────────────────
REDESIGN_DIR = Path('redesign/outputs')
MPNN_DIR = Path('/tmp/ProteinMPNN')

print('ProteinMPNN script exists:', (MPNN_DIR / 'protein_mpnn_run.py').exists())
print()

# Show all files written during the 100-structure run
all_files = list(REDESIGN_DIR.rglob('*'))
print(f'Total files in redesign/outputs: {len(all_files)}')
for f in sorted(all_files)[:30]:   # show first 30
    print(f'  {f}')
if len(all_files) > 30:
    print(f'  ... and {len(all_files)-30} more')

ProteinMPNN script exists: True

Total files in redesign/outputs: 0


In [3]:
# ── Show the first FASTA file's content ──────────────────────
fa_files = list(REDESIGN_DIR.rglob('*.fa'))
print(f'Total .fa files: {len(fa_files)}')

if fa_files:
    print(f'\nFirst file: {fa_files[0]}')
    print('--- Content (first 20 lines) ---')
    with open(fa_files[0]) as f:
        for i, line in enumerate(f):
            if i >= 20: break
            print(repr(line.rstrip()))
else:
    print('No .fa files found — checking stderr of a fresh single run...')
    with open('data/structures_manifest.json') as f:
        manifest = json.load(f)
    if manifest:
        entry = manifest[0]
        out_dir = REDESIGN_DIR / Path(entry['pdb_path']).stem
        out_dir.mkdir(exist_ok=True)
        script = MPNN_DIR / 'protein_mpnn_run.py'
        result = subprocess.run(
            ['python', str(script),
             '--pdb_path', entry['pdb_path'],
             '--out_folder', str(out_dir),
             '--num_seq_per_target', '3',
             '--sampling_temp', '0.1',
             '--seed', '37', '--batch_size', '1'],
            capture_output=True, timeout=60
        )
        print('Return code:', result.returncode)
        print('--- stdout ---')
        print(result.stdout.decode()[:1000])
        print('--- stderr ---')
        print(result.stderr.decode()[:1000])
        print('--- files written ---')
        for f in out_dir.rglob('*'):
            print(f'  {f}')

Total .fa files: 0
No .fa files found — checking stderr of a fresh single run...
Return code: 0
--- stdout ---
----------------------------------------
chain_id_jsonl is NOT loaded
----------------------------------------
fixed_positions_jsonl is NOT loaded
----------------------------------------
pssm_jsonl is NOT loaded
----------------------------------------
omit_AA_jsonl is NOT loaded
----------------------------------------
bias_AA_jsonl is NOT loaded
----------------------------------------
tied_positions_jsonl is NOT loaded
----------------------------------------
bias by residue dictionary is not loaded, or not provided
----------------------------------------
----------------------------------------
Number of edges: 48
Training noise level: 0.2A
Generating sequences for: P0DJJ1
3 sequences of length 16 generated in 0.2433 seconds

--- stderr ---

--- files written ---
  redesign/outputs/P0DJJ1/seqs
  redesign/outputs/P0DJJ1/seqs/P0DJJ1.fa


## 4. ESM-2 Embedding Extraction

In [4]:
from transformers import AutoTokenizer, EsmForMaskedLM

MODEL_NAME = 'facebook/esm2_t33_650M_UR50D'
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
esm2 = EsmForMaskedLM.from_pretrained(
    MODEL_NAME, output_hidden_states=True
).to(DEVICE).eval()
print('Model loaded.')


def extract_embeddings(records, layers, batch_size=8, max_len=1022):
    layer_embs = {l: [] for l in layers}
    for i in tqdm(range(0, len(records), batch_size), desc='Extracting'):
        batch = records[i:i + batch_size]
        seqs = [str(r.seq)[:max_len] for r in batch]
        inputs = tokenizer(seqs, return_tensors='pt', padding=True,
                            truncation=True, max_length=max_len + 2)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = esm2(**inputs)
            hidden = outputs.hidden_states
        attn_mask = inputs['attention_mask'].unsqueeze(-1).float()
        for layer in layers:
            h = hidden[layer]
            pooled = (h * attn_mask).sum(1) / attn_mask.sum(1)
            layer_embs[layer].append(pooled.cpu().float().numpy())
    return {l: np.concatenate(v, axis=0) for l, v in layer_embs.items()}


tox_reps  = list(SeqIO.parse('data/toxins_clustered.fasta',  'fasta'))
ctrl_reps = list(SeqIO.parse('data/controls_clustered.fasta', 'fasta'))

print(f'Extracting {len(tox_reps)} toxins...')
tox_embs = extract_embeddings(tox_reps, LAYERS)

print(f'Extracting {len(ctrl_reps)} controls...')
ctrl_embs = extract_embeddings(ctrl_reps, LAYERS)

for layer in LAYERS:
    np.save(f'embeddings/natural_toxins_layer{layer}.npy', tox_embs[layer])
    np.save(f'embeddings/controls_layer{layer}.npy', ctrl_embs[layer])
    print(f'  Layer {layer}: tox={tox_embs[layer].shape}, ctrl={ctrl_embs[layer].shape}')
print('Embeddings saved.')

Loading facebook/esm2_t33_650M_UR50D...


Loading weights: 100%|██████████| 539/539 [00:00<00:00, 6467.64it/s]


Model loaded.
Extracting 1712 toxins...


Extracting: 100%|██████████| 214/214 [01:00<00:00,  3.52it/s]


Extracting 2072 controls...


Extracting: 100%|██████████| 259/259 [01:44<00:00,  2.47it/s]

  Layer 1: tox=(1712, 1280), ctrl=(2072, 1280)
  Layer 9: tox=(1712, 1280), ctrl=(2072, 1280)
  Layer 18: tox=(1712, 1280), ctrl=(2072, 1280)
  Layer 24: tox=(1712, 1280), ctrl=(2072, 1280)
  Layer 30: tox=(1712, 1280), ctrl=(2072, 1280)
  Layer 33: tox=(1712, 1280), ctrl=(2072, 1280)
Embeddings saved.


## 5. ProteinMPNN Redesign (fixed output parsing)

In [5]:
MPNN_DIR     = Path('/tmp/ProteinMPNN')
REDESIGN_DIR = Path('redesign/outputs')
REDESIGN_DIR.mkdir(parents=True, exist_ok=True)


def parse_mpnn_fasta(out_dir: Path) -> list[str]:
    """
    ProteinMPNN writes sequences to out_folder/seqs/{stem}.fa
    NOT out_folder/*.fa

    FASTA format:
      >native_seq, score=-1.23, ...      ← native (skip)
      >T=0.1, sample=1, score=-1.45, ... ← designed (keep)
    """
    seqs = []
    # Primary location: seqs/ subdirectory
    for fa in list(out_dir.glob('seqs/*.fa')) + list(out_dir.glob('*.fa')):
        for rec in SeqIO.parse(str(fa), 'fasta'):
            desc = rec.description
            # Skip native sequence (first record, no T= tag)
            if 'T=' in desc or 'sample=' in desc:
                seq_str = str(rec.seq).replace('/', '')  # remove chain breaks
                if len(seq_str) > 0:
                    seqs.append(seq_str)
    return seqs


def run_proteinmpnn(pdb_path: str, n_seqs: int = 10,
                     temperature: float = 0.1) -> list[str]:
    script = MPNN_DIR / 'protein_mpnn_run.py'
    if not script.exists():
        print(f'  ERROR: {script} not found')
        return []

    stem    = Path(pdb_path).stem
    out_dir = REDESIGN_DIR / stem
    out_dir.mkdir(exist_ok=True)

    # Skip if already done
    existing = parse_mpnn_fasta(out_dir)
    if len(existing) >= n_seqs:
        return existing

    cmd = [
        'python', str(script),
        '--pdb_path',           pdb_path,
        '--out_folder',         str(out_dir),
        '--num_seq_per_target', str(n_seqs),
        '--sampling_temp',      str(temperature),
        '--seed',               '37',
        '--batch_size',         '1',
    ]
    result = subprocess.run(cmd, capture_output=True, timeout=180)

    if result.returncode != 0:
        print(f'  MPNN error for {stem}: {result.stderr.decode()[:200]}')
        return []

    return parse_mpnn_fasta(out_dir)


# ── Run over all structures ───────────────────────────────────
with open('data/structures_manifest.json') as f:
    manifest = json.load(f)

all_redesigns = []
for entry in tqdm(manifest, desc='ProteinMPNN'):
    seqs = run_proteinmpnn(entry['pdb_path'], n_seqs=10)
    for s in seqs:
        all_redesigns.append({
            'sequence':   s,
            'uniprot_id': entry['uniprot_id'],
            'source_pdb': entry['pdb_path'],
        })

print(f'Total redesigns: {len(all_redesigns)}')
print(f'Expected:        {len(manifest) * 10} (100 structures × 10 seqs)')

ProteinMPNN: 100%|██████████| 100/100 [05:35<00:00,  3.35s/it]

Total redesigns: 1000
Expected:        1000 (100 structures × 10 seqs)


In [6]:
# ── Identity filter ───────────────────────────────────────────
train_toxins = [str(r.seq) for r in SeqIO.parse('data/toxins_clustered.fasta', 'fasta')]

def max_identity_to_set(query: str, refs: list, sample: int = 200) -> float:
    best = 0.0
    for ref in refs[:sample]:
        q, s = query[:150], ref[:150]
        score = pairwise2.align.globalxx(q, s, score_only=True)
        best = max(best, score / max(len(q), len(s)))
    return best


THRESHOLD = 0.40
filtered, identities = [], []
for d in tqdm(all_redesigns, desc='Identity filter'):
    ident = max_identity_to_set(d['sequence'], train_toxins)
    identities.append(ident)
    if ident < THRESHOLD:
        d['max_identity_to_training'] = ident
        filtered.append(d)

print(f'Passed <{THRESHOLD*100:.0f}%: {len(filtered)}/{len(all_redesigns)}')

if len(filtered) < 200:
    THRESHOLD = 0.50
    filtered = [{**d, 'max_identity_to_training': id_}
                for d, id_ in zip(all_redesigns, identities) if id_ < THRESHOLD]
    print(f'Relaxed to <{THRESHOLD*100:.0f}%: {len(filtered)}')

with open('redesign/outputs/redesign_metadata.json', 'w') as f:
    json.dump(filtered, f, indent=2)

redesign_records = [
    SeqRecord(Seq(d['sequence']),
              id=f"{d['uniprot_id']}_r{i}",
              description=f"identity={d['max_identity_to_training']:.3f}")
    for i, d in enumerate(filtered)
]
SeqIO.write(redesign_records, 'redesign/outputs/redesigns.fasta', 'fasta')
print(f'Saved {len(redesign_records)} redesigns → redesign/outputs/redesigns.fasta')

Identity filter: 100%|██████████| 1000/1000 [00:06<00:00, 164.97it/s]

Passed <40%: 643/1000
Saved 643 redesigns → redesign/outputs/redesigns.fasta


In [7]:
# ── Embed redesigns ───────────────────────────────────────────
redesign_recs = list(SeqIO.parse('redesign/outputs/redesigns.fasta', 'fasta'))
print(f'Embedding {len(redesign_recs)} redesigns...')

redesign_embs = extract_embeddings(redesign_recs, LAYERS)

for layer in LAYERS:
    np.save(f'embeddings/redesigns_layer{layer}.npy', redesign_embs[layer])
    print(f'  Layer {layer}: {redesign_embs[layer].shape}')
print('Redesign embeddings saved.')

Embedding 643 redesigns...


Extracting: 100%|██████████| 81/81 [00:02<00:00, 34.29it/s]

  Layer 1: (643, 1280)
  Layer 9: (643, 1280)
  Layer 18: (643, 1280)
  Layer 24: (643, 1280)
  Layer 30: (643, 1280)
  Layer 33: (643, 1280)
Redesign embeddings saved.


## 6. Sequence Identity Arrays for Generalization Curve

In [8]:
from sklearn.model_selection import train_test_split

tox_recs  = list(SeqIO.parse('data/toxins_clustered.fasta',  'fasta'))
ctrl_recs = list(SeqIO.parse('data/controls_clustered.fasta', 'fasta'))
tox_seqs  = [str(r.seq) for r in tox_recs]
ctrl_seqs = [str(r.seq) for r in ctrl_recs]

all_seqs   = tox_seqs + ctrl_seqs
all_labels = [1]*len(tox_seqs) + [0]*len(ctrl_seqs)
idx = list(range(len(all_seqs)))
train_idx, test_idx = train_test_split(idx, test_size=0.2,
                                        stratify=all_labels, random_state=42)

train_tox_seqs = [all_seqs[i] for i in train_idx if all_labels[i] == 1]
test_seqs_nat  = [all_seqs[i] for i in test_idx]
redesign_seqs  = [str(r.seq) for r in redesign_recs]
all_test_seqs  = test_seqs_nat + redesign_seqs

print(f'Test: {len(test_seqs_nat)} natural + {len(redesign_seqs)} redesigns')

seq_identities = np.array([
    max_identity_to_set(seq, train_tox_seqs, sample=300)
    for seq in tqdm(all_test_seqs, desc='Seq identity')
])
np.save('embeddings/sequence_identities.npy', seq_identities)
print(f'Identity: mean={seq_identities.mean():.3f}, '
      f'min={seq_identities.min():.3f}, max={seq_identities.max():.3f}')

# BLAST baseline
SeqIO.write(
    [SeqRecord(Seq(s), id=f'train_{i}', description='') for i, s in enumerate(train_tox_seqs)],
    '/tmp/train_toxins.fasta', 'fasta'
)
blast_ok = subprocess.run(
    'makeblastdb -in /tmp/train_toxins.fasta -dbtype prot -out /tmp/toxin_db',
    shell=True, capture_output=True
).returncode == 0

if blast_ok:
    SeqIO.write(
        [SeqRecord(Seq(s), id=f'test_{i}', description='') for i, s in enumerate(all_test_seqs)],
        '/tmp/test_seqs.fasta', 'fasta'
    )
    subprocess.run(
        'blastp -query /tmp/test_seqs.fasta -db /tmp/toxin_db '
        '-out /tmp/blast_results.xml -outfmt 5 -evalue 1e-3 -num_threads 8 -max_hsps 1',
        shell=True
    )
    blast_identities = np.zeros(len(all_test_seqs))
    from Bio.Blast import NCBIXML
    with open('/tmp/blast_results.xml') as f:
        for i, rec in enumerate(NCBIXML.parse(f)):
            if rec.alignments:
                hsp = rec.alignments[0].hsps[0]
                blast_identities[i] = hsp.identities / hsp.align_length
    print('BLAST done.')
else:
    print('BLAST+ unavailable — using pairwise identity as proxy.')
    blast_identities = seq_identities.copy()

np.save('embeddings/blast_identities.npy', blast_identities)
print('All identity arrays saved.')

Test: 757 natural + 643 redesigns


Seq identity: 100%|██████████| 1400/1400 [00:21<00:00, 64.34it/s] 


Identity: mean=0.394, min=0.158, max=0.800
BLAST done.
All identity arrays saved.
